# Silver Layer — Data Cleaning

This notebook prepares the raw vehicle listings for analysis.

The data is read from the Bronze layer, checked for quality, and cleaned before being saved to the Silver layer.

**Source:** `car_sales.bronze.vehicles_raw`

**Output:** `car_sales.silver.vehicles_clean`

**Main steps**
- Check the source data.
- Correct column data types.
- Review missing values and duplicate listings.
- Remove invalid records.
- Save the cleaned data.
- Validate the final output.

In [0]:
from pyspark.sql import functions as F

df_bronze = spark.table("car_sales.bronze.vehicles_raw")

print(f"Source row count: {df_bronze.count():,}")
print(f"Number of columns: {len(df_bronze.columns)}")

Source row count: 426,880
Number of columns: 28


## 1. Inspect Data Quality

Before cleaning the data, we will check for missing values and duplicate listings.

This helps us understand which fields need attention and avoid removing useful records unnecessarily.

In [0]:
missing_values = df_bronze.select(
    *[
        F.count(
            F.when(
                F.col(column).isNull() |
                (F.trim(F.col(column).cast("string")) == ""),
                column
            )
        ).alias(column)
        for column in df_bronze.columns
    ]
)

display(missing_values)

id,url,region,region_url,price,year,manufacturer,model,condition,cylinders,fuel,odometer,title_status,transmission,VIN,drive,size,type,paint_color,image_url,description,county,state,lat,long,posting_date,_ingestion_timestamp,_source_file
0,0,0,0,0,1205,17646,5276,174104,177678,3013,4400,8242,2556,161042,130567,306361,92858,130203,68,69,426880,0,6549,6549,68,0,0


## 2. Check Duplicate Listings

We will check whether the source contains repeated listing IDs.

The listing ID will be used to identify duplicate records before we clean the data.

In [0]:
duplicate_ids = (
    df_bronze
    .groupBy("id")
    .count()
    .filter(F.col("count") > 1)
)

print(f"Duplicate listing IDs: {duplicate_ids.count():,}")

Duplicate listing IDs: 0


## 3. Prepare the Data

Some columns, such as price, model year, and odometer reading, were loaded as text.

We will convert them to the correct data types so that calculations and filtering work correctly. We will also standardize blank values as nulls.

In [0]:
df_silver = df_bronze

for column in df_silver.columns:
    if df_silver.schema[column].dataType.simpleString() == "string":
        df_silver = df_silver.withColumn(
            column,
            F.when(
                F.trim(F.col(column)) == "",
                None
            ).otherwise(F.trim(F.col(column)))
        )

df_silver = (
    df_silver
    .withColumn("price", F.expr("try_cast(price AS DOUBLE)"))
    .withColumn("year", F.expr("try_cast(year AS INT)"))
    .withColumn("odometer", F.expr("try_cast(odometer AS DOUBLE)"))
    .withColumn(
        "cylinders",
        F.expr(
            "try_cast(regexp_extract(cylinders, '[0-9]+', 0) AS INT)"
        )
    )
    .withColumn("lat", F.expr("try_cast(lat AS DOUBLE)"))
    .withColumn("long", F.expr("try_cast(long AS DOUBLE)"))
)

print("Data preparation completed.")
df_silver.printSchema()

Data preparation completed.
root
 |-- id: string (nullable = true)
 |-- url: string (nullable = true)
 |-- region: string (nullable = true)
 |-- region_url: string (nullable = true)
 |-- price: double (nullable = true)
 |-- year: integer (nullable = true)
 |-- manufacturer: string (nullable = true)
 |-- model: string (nullable = true)
 |-- condition: string (nullable = true)
 |-- cylinders: integer (nullable = true)
 |-- fuel: string (nullable = true)
 |-- odometer: double (nullable = true)
 |-- title_status: string (nullable = true)
 |-- transmission: string (nullable = true)
 |-- VIN: string (nullable = true)
 |-- drive: string (nullable = true)
 |-- size: string (nullable = true)
 |-- type: string (nullable = true)
 |-- paint_color: string (nullable = true)
 |-- image_url: string (nullable = true)
 |-- description: string (nullable = true)
 |-- county: string (nullable = true)
 |-- state: string (nullable = true)
 |-- lat: double (nullable = true)
 |-- long: double (nullable = true)

## 4. Validate Vehicle Records

We will review the main numeric fields to identify invalid values.

Prices must be positive, model years must be reasonable, and odometer readings must not be negative. We will inspect these values before removing any records.

In [0]:
quality_checks = df_silver.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("price").isNull(), 1).otherwise(0)).alias("missing_price"),
    F.sum(F.when(F.col("price") <= 0, 1).otherwise(0)).alias("invalid_price"),
    F.sum(F.when(F.col("year").isNull(), 1).otherwise(0)).alias("missing_year"),
    F.sum(
        F.when(
            (F.col("year") < 1900) | (F.col("year") > 2027),
            1
        ).otherwise(0)
    ).alias("invalid_year"),
    F.sum(
        F.when(F.col("odometer") < 0, 1).otherwise(0)
    ).alias("negative_odometer")
)

display(quality_checks)

total_rows,missing_price,invalid_price,missing_year,invalid_year,negative_odometer
426880,0,32895,1205,0,0


## 5. Apply Cleaning Rules

We will remove records that cannot be used reliably for vehicle price analysis.

- Keep listings with a positive price.
- Keep listings with a valid model year between 1900 and 2027.
- Keep missing odometer readings, but exclude negative readings.

These rules retain useful listings while removing records with invalid values.

In [0]:
df_silver = df_silver.filter(
    (F.col("price") > 0) &
    F.col("year").between(1900, 2027) &
    (F.col("odometer").isNull() | (F.col("odometer") >= 0))
)

print(f"Rows after cleaning: {df_silver.count():,}")

Rows after cleaning: 392,812


## 6. Review Cleaned Data

We will review the cleaned records to confirm that the price and model year values are valid.

We will also check the remaining missing values before saving the Silver table.

In [0]:
display(
    df_silver.select(
        F.count("*").alias("total_rows"),
        F.min("price").alias("minimum_price"),
        F.max("price").alias("maximum_price"),
        F.min("year").alias("earliest_model_year"),
        F.max("year").alias("latest_model_year"),
        F.sum(F.when(F.col("price").isNull(), 1).otherwise(0)).alias("missing_price"),
        F.sum(F.when(F.col("year").isNull(), 1).otherwise(0)).alias("missing_year")
    )
)

total_rows,minimum_price,maximum_price,earliest_model_year,latest_model_year,missing_price,missing_year
392812,1.0,3.736928711E9,1900,2022,0,0


## 7. Save the Cleaned Data

The cleaned vehicle listings are ready to be saved.

We will store them as a Delta table in the Silver schema so that the Gold layer can use them for reporting and analysis.

In [0]:
(
    df_silver.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("car_sales.silver.vehicles_clean")
)

print("Silver table saved successfully.")

Silver table saved successfully.


## 8. Verify the Silver Table

We will compare the cleaned data with the saved Silver table to confirm that all cleaned records have been stored correctly.

We will also preview a few records to verify the output.

In [0]:
df_silver_saved = spark.table("car_sales.silver.vehicles_clean")

print(f"Cleaned DataFrame rows: {df_silver.count():,}")
print(f"Saved Silver table rows: {df_silver_saved.count():,}")

display(df_silver_saved.limit(10))

Cleaned DataFrame rows: 392,812
Saved Silver table rows: 392,812


id url region region_url price year manufacturer model condition cylinders fuel odometer title_status transmission VIN drive size type paint_color image_url description county state lat long posting_date _ingestion_timestamp _source_file 7308745660 https://zanesville.craigslist.org/ctd/d/zanesville-2011-dodge-challenger-srt8/7308745660.html zanesville / cambridge https://zanesville.craigslist.org 35590.0 2011 dodge challenger srt8 coupe 2d good 8 gas 16796.0 clean other 2B3CJ7DJ7BH589702 rwd null coupe green https://images.craigslist.org/00P0P_25B5uUcC4Bwz_0gw0co_600x450.jpg Carvana is the safer way to buy a car During these uncertain times, Carvana is dedicated to ensuring safety for all of our customers. In addition to our 100% online shopping and selling experience that allows all customers to buy and trade their cars without ever leaving the safety of their house, we’re providing touchless delivery that make all aspects of our process even safer. Now, you can get the car you want, and trade in your old one, while avoiding person-to-person contact with our friendly advocates. There are some things that can’t be put off. And if buying a car is one of them, know that we’re doing everything we can to keep you keep moving while continuing to put your health safety, and happiness first. Vehicle Stock# 2000937990📱 Want to instantly check this car’s availability? Call us at 740-400-0350Just text that stock number to 855-976-4304 or head to and plug it into the search bar!Get PRE-QUALIFIED for your auto loan in 2 minutes - no hit to your credit:http://finance.carvanaauto.com/6965054-74736Looking for more cars like this one? We have 158 Dodge Challenger in stock for as low as $15990!Why buy with Carvana? We have one standard: the highest. Take a look at just some of the qualifications all of our cars must meet before we list them.150-POINT INSPECTION: We put each vehicle through a 150-point inspection so that you can be 100% confident in its quality and safety. See everything that goes into our inspections at:NO REPORTED ACCIDENTS: We do not sell cars that have been in a reported accident or have a frame or structural damage.7 DAY TEST OWN MONEY BACK GUARANTEE: Every Carvana car comes with a 7-day money-back guarantee. Why? It takes more than 15-minutes to make a decision on your next car. Learn more about test owning at http://about.carvanaauto.comFLEXIBLE FINANCING, TRADE INS WELCOME: We’re all about real-time financing without the middle man. Need financing? Pick a combination of down and monthly payments that work for you. Have a trade-in? We’ll give you a value in 2 minutes. Check out everything about our financing at:http://finance.carvanaauto.com/6965054-74736COST SAVINGS: Carvana's business model has fewer expenses and no bloated fees compared to your local dealership. See how much we can save you at http://about.carvanaauto.comPREMIUM DETAIL: We go the extra mile so that your car is looking as good as new. There are a lot of specifics that we won’t list here (we wash, clean, buff, paint, polish, wax, seal), but trust us that when your car arrives, it’s going to look sweet.Vehicle Info for Stock# 2000937990Trim: SRT8 Coupe 2D coupeMileage: 16k milesExterior Color: GreenInterior Color: GrayEngine: HEMI 6.4L V8 470hp 470ft. lbs.Drive: rwdTransmission: VIN: 2B3CJ7DJ7BH589702Dealer Disclosure: Price excludes tax, title, and registration (which we handle for you).Disclaimer: You agree that by providing your phone number, Carvana, or Carvana’s authorized representatives*, may call and/or send text messages (including by using equipment to automatically dial telephone numbers) about your interest in a purchase, for marketing/sales purposes, or for any other servicing or informational purpose related to your account. You do not have to consent to receiving calls or texts to purchase from Carvana. While every reasonable effort is made to ensure the accuracy of the information for this Dodge Challenger, we are not responsible for any 